# Fixed LightGBM on preserved causal factor features
Attach version10 full-row notebook outputs, causal feature DataSet, and neural-history dataset. No factor retraining. Fixed 800 rounds, matched 354/371 feature arms, fixed10% rank blend. Runtime capability probe chooses GPU if supported, otherwise Kaggle CPU4threads. Estimated10–25minutes. Reused-cohort development only; no CSV export.

In [ ]:
from pathlib import Path
import importlib.util,hashlib
p=Path("/kaggle/working/tied_history_lgbm.py")
p.write_text('"""Fixed LightGBM comparison using preserved causal OOF factors, without refitting factors."""\nimport gc\nimport hashlib\nimport importlib.util\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport lightgbm as lgb\nfrom scipy.stats import rankdata\nfrom sklearn.metrics import roc_auc_score\n\nPARAMS = dict(objective=\'binary\', learning_rate=.05, num_leaves=31, max_depth=6,\n              min_sum_hessian_in_leaf=50, lambda_l2=5., lambda_l1=.1,\n              feature_fraction=.9, bagging_fraction=.8, bagging_freq=1,\n              max_bin=63, num_threads=4, seed=7502, verbosity=-1)\nROUNDS = 800\nBLEND = .1\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(4*1024*1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef run():\n    started = time.monotonic()\n    roots = list(Path(\'/kaggle/input\').rglob(\'bundle_manifest.json\'))\n    neural = list(Path(\'/kaggle/input\').rglob(\'neural_manifest.json\'))\n    previous = list(Path(\'/kaggle/input\').rglob(\'trees6/predictions.npz\'))\n    assert len(roots) == len(neural) == len(previous) == 1\n    root, nr, prior = roots[0].parent, neural[0].parent, previous[0].parent.parent\n    out = Path(\'/kaggle/working/tied-history-lgbm-001\')\n    out.mkdir(exist_ok=False)\n    # Probe only runtime capability; no validation labels or model settings selected.\n    device = \'cpu\'\n    try:\n        probe = lgb.Dataset(np.arange(400, dtype=\'float32\').reshape(100,4), label=np.arange(100)%2)\n        lgb.train(dict(PARAMS, device_type=\'gpu\'), probe, num_boost_round=1)\n        device = \'gpu\'\n    except lgb.basic.LightGBMError as error:\n        print(json.dumps(dict(stage=\'gpu_probe\', fallback=\'cpu\', reason=str(error))), flush=True)\n    params = dict(PARAMS, device_type=device)\n    frozen = dict(params=params, rounds=ROUNDS, blend=BLEND, factor_retraining=False,\n                  source_sha256=digest(__file__), prior_prediction_sha256=digest(previous[0]),\n                  expected_prior_prediction_sha256=\'3ad63970946c7eb4b0e5f51efa9ad4a761a90bee93ab0d79cbb22c24449198ff\',\n                  fresh_holdout=False, promoted=False, official_auc=None)\n    assert frozen[\'prior_prediction_sha256\'] == frozen[\'expected_prior_prediction_sha256\']\n    (out/\'frozen.json\').write_text(json.dumps(frozen, indent=2))\n    print(json.dumps(frozen), flush=True)\n    spec = importlib.util.spec_from_file_location(\'causal_features\', root/\'runner.py\')\n    helpers = importlib.util.module_from_spec(spec);spec.loader.exec_module(helpers)\n    manifest = helpers.load_bundle(root, verify=True)\n    with np.load(nr/\'events.npz\') as z:\n        hist, target = z[\'hist\'], z[\'target\']\n    with np.load(nr/\'val6.npz\') as z:\n        groups = np.digitize((hist[z[\'indices\']]>=0).sum(1), [0,4,9,19], right=True)\n        source = np.bincount(groups,minlength=5)/len(groups)\n        w = z[\'weights\'].reshape(-1,10)[:,0]\n        shares = source*np.array([w[groups==g][0] for g in range(5)])\n        assert np.isclose(shares.sum(),1)\n    specs = [manifest[\'train\'][str(d)] for d in [3,4,5]]\n    labels, ids, extra = [], [], []\n    for d,s in zip([3,4,5],specs):\n        factor = prior/f\'factors{d}\'\n        report = json.loads((factor/\'report.json\').read_text())\n        assert report[\'factor_fit_last_timestamp\'] < report[\'cutoff\'] <= report[\'score_first_timestamp\']\n        assert digest(factor/\'features.npy\') == report[\'feature_sha256\']\n        dayids = np.load(factor/\'event_ids.npy\')\n        y = np.load(root/s[\'labels\'])\n        assert np.array_equal(y,(target[dayids,None]==np.arange(10)).ravel())\n        labels.append(y);ids.append(dayids);extra.append(np.load(factor/\'features.npy\',mmap_mode=\'r\'))\n    labels = np.concatenate(labels).astype(\'float32\');ids=np.concatenate(ids)\n    groups = np.digitize((hist[ids]>=0).sum(1),[0,4,9,19],right=True)\n    source=np.bincount(groups,minlength=5)/len(groups)\n    weights=np.repeat((shares/source)[groups],10).astype(\'float32\');weights/=weights.mean()\n    base=helpers.FeatureBlocks(root,[s[\'features\'] for s in specs])\n    assert len(base)==len(labels)==2937590\n    extra=np.concatenate(extra)\n    assert extra.shape==(len(labels),17)\n    dense=np.lib.format.open_memmap(out/\'training_features.npy\',mode=\'w+\',dtype=\'float32\',shape=(len(labels),371))\n    for off in range(0,len(labels),100000):\n        end=min(off+100000,len(labels));dense[off:end,:354]=base.block(off,end);dense[off:end,354:]=extra[off:end]\n    dense.flush();del extra,hist,target,ids;gc.collect()\n    validation=manifest[\'validation\'][\'6\'];val=np.load(root/validation[\'data\'])\n    old=np.load(previous[0])\n    for key in [\'y\',\'events\',\'weights\',\'baseline\']:\n        np.testing.assert_allclose(val[key],old[key],rtol=0,atol=1e-12)\n    vb=helpers.FeatureBlocks(root,[validation[\'features\']])\n    vx=np.empty((len(vb),371),dtype=\'float32\')\n    for off in range(0,len(vb),100000):\n        end=min(off+100000,len(vb));vx[off:end,:354]=vb.block(off,end)\n    vx[:,354:]=np.load(prior/\'trees6/validation_extra.npy\')\n    predictions={}\n    for arm,width in [(\'control\',354),(\'treatment\',371)]:\n        print(json.dumps(dict(stage=\'tree_start\',arm=arm,rows=len(labels),features=width)),flush=True)\n        dataset=lgb.Dataset(dense[:,:width],label=labels,weight=weights,free_raw_data=True,\n                            params={\'max_bin\':63})\n        def progress(env):\n            if (env.iteration+1)%100==0:\n                print(json.dumps(dict(stage=\'tree_training\',arm=arm,round=env.iteration+1,total=ROUNDS)),flush=True)\n        model=lgb.train(params,dataset,num_boost_round=ROUNDS,callbacks=[progress])\n        assert model.current_iteration()==ROUNDS\n        model.save_model(str(out/f\'{arm}.txt\'))\n        raw=model.predict(vx[:,:width],num_threads=4)\n        replay=lgb.Booster(model_file=str(out/f\'{arm}.txt\'))\n        assert np.array_equal(raw,replay.predict(vx[:,:width],num_threads=4))\n        rank=lambda p:rankdata(np.asarray(p,dtype=\'float64\'),method=\'average\')/(len(p)+1)\n        predictions[arm+\'_raw\']=raw\n        predictions[arm]=(1-BLEND)*rank(val[\'baseline\'])+BLEND*rank(raw)\n        del model,replay,dataset;gc.collect()\n    def metric(p):\n        return dict(ordinary_auc=float(roc_auc_score(val[\'y\'],p)),\n                    adjusted_auc=float(roc_auc_score(val[\'y\'],p,sample_weight=val[\'weights\'])))\n    report=dict(baseline=metric(val[\'baseline\']),**{k:metric(v) for k,v in predictions.items()},\n                factor_retraining=False,training_rows=len(labels),elapsed_seconds=time.monotonic()-started,\n                fresh_holdout=False,promoted=False,csv_ready=False,official_auc=None,target_achieved=False)\n    report[\'adjusted_gain\']=report[\'treatment\'][\'adjusted_auc\']-report[\'baseline\'][\'adjusted_auc\']\n    report[\'gain_vs_control\']=report[\'treatment\'][\'adjusted_auc\']-report[\'control\'][\'adjusted_auc\']\n    report[\'development_gate\']=(report[\'adjusted_gain\']>=.0005 and report[\'gain_vs_control\']>0\n                                and report[\'treatment\'][\'ordinary_auc\']>=report[\'baseline\'][\'ordinary_auc\'])\n    np.savez_compressed(out/\'predictions.npz\',y=val[\'y\'],events=val[\'events\'],weights=val[\'weights\'],\n                        baseline=val[\'baseline\'],**predictions)\n    (out/\'report.json\').write_text(json.dumps(report,indent=2))\n    print(json.dumps(report),flush=True)\n    # Keep the saved models and predictions; remove only our disposable design matrix.\n    del dense;gc.collect()\n    (out/\'training_features.npy\').unlink()\n    return report\n',encoding="utf-8")
assert hashlib.sha256(p.read_bytes()).hexdigest()=='2e4317658716c1e5c9474caadcc37614c19a8136ce9fef0278ee2630d2ce7edc'
spec=importlib.util.spec_from_file_location("tied_history_lgbm",p)
module=importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
result=module.run()
